In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

c:\Users\Furqan Khan\AppData\Local\miniconda3\envs\agent_env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define the directories

In [2]:
base_dir = os.path.join(".", "images_dataSAT")

dir_non_agri = os.path.join(
    base_dir,
    "class_0_non_agri"
)

dir_agri = os.path.join(
    base_dir,
    "class_1_agri"
)

# Custom PyTorch Dataset
- __init__()
- __len__()
- __getitem__()

Dataset
   ↓
one image + one label
   ↓
DataLoader
   ↓
batch of images + labels

In [3]:
class CustomDataset(Dataset):

    def __init__(
        self,
        non_agri_dir,
        agri_dir,
        transform=None
    ):
        self.transform = transform
        self.images = []
        self.labels = []

        for file in os.listdir(non_agri_dir):
            self.images.append(
                os.path.join(non_agri_dir, file)
            )
            self.labels.append(0)

        for file in os.listdir(agri_dir):
            self.images.append(
                os.path.join(agri_dir, file)
            )
            self.labels.append(1)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = Image.open(
            self.images[index]
        ).convert("RGB")

        label = self.labels[index]

        if self.transform:
            image = self.transform(image)

        return image, label

# Transformations
- Resize
- RandomHorizontalFlip
- RandomVerticalFlip
- RandomRotation
- ToTensor
- Normalize

PIL image
   ↓
Resize
   ↓
Augmentation
   ↓
ToTensor
   ↓
Normalize
   ↓
Tensor

In [ ]:
custom_transform = transforms.Compose([
    transforms.Resize((64, 64)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.2),

    transforms.RandomRotation(45),

    transforms.ToTensor(),
# *3 means for the three RGB channels.
    transforms.Normalize(
        [0.5] * 3,
        [0.5] * 3
    ),
])

custom_transform

Compose(
    Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.2)
    RandomRotation(degrees=[-45.0, 45.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
)

## So ImageFolder can automatically discover:

images
+
classes

instead of requiring you to write the CustomDataset class.

# IMPORTANT:ImageFolder only works when images are organized in class folders. CustomDataset gives full control and can load data from CSV files, databases, APIs, or any custom structure.

In [5]:
imagefolder_dataset = datasets.ImageFolder(
    root=base_dir,
    transform=custom_transform,
)

imagefolder_dataset

Dataset ImageFolder
    Number of datapoints: 6000
    Root location: .\images_dataSAT
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               RandomHorizontalFlip(p=0.5)
               RandomVerticalFlip(p=0.2)
               RandomRotation(degrees=[-45.0, 45.0], interpolation=nearest, expand=False, fill=0)
               ToTensor()
               Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
           )

In [6]:
print(
    "Classes:",
    imagefolder_dataset.classes
)

print(
    "Class indexes:",
    imagefolder_dataset.class_to_idx
)

Classes: ['class_0_non_agri', 'class_1_agri']
Class indexes: {'class_0_non_agri': 0, 'class_1_agri': 1}


# DataLoader
return 8 images
8 labels


B = batch size
C = channels
H = height
W = width

In [7]:
BATCH_SIZE = 8

custom_dataset = CustomDataset(
    dir_non_agri,
    dir_agri,
    transform=custom_transform,
)

custom_loader = DataLoader(
    custom_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
)

imagefolder_loader = DataLoader(
    imagefolder_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
)

print("DataLoaders created successfully!")

DataLoaders created successfully!


custom_loader
      ↓

CustomDataset

imagefolder_loader
      ↓
      
ImageFolder

### **Task 4**: Get a batch of images and labels from the `imagefolder_loader` and print their shape

In [8]:
images_inbuilt, labels_inbuilt = next(
    iter(imagefolder_loader)
)

print("Images:", images_inbuilt.shape)
print("Labels:", labels_inbuilt.shape)

Images: torch.Size([8, 3, 64, 64])
Labels: torch.Size([8])


Next, define a function to display an image from the batch.

In [9]:
def imshow(image):
    """Display an image after undoing [-1, 1] normalization."""

    image = image / 2 + 0.5

    plt.imshow(
        image.permute(1, 2, 0).numpy()
    )

    plt.axis("off")

[3, 64, 64]
      ↓
[64, 64, 3]

### **Task 5**: Display the images in the Custom loader batch

Similar to the code cell above, display the images stored in `images_custom` generated using `custom_loader`.

The title of the images should be **`Custom_loader Label: `** similar to the images seen in the above cell

In [ ]:
images_custom, labels_custom = next(
    iter(custom_loader)
)

plt.figure(figsize=(12, 6))

for i in range(BATCH_SIZE):
    ax = plt.subplot(2, 4, i + 1)

    imshow(images_custom[i])

    plt.title(
        f"Custom_loader Label: {labels_custom[i].item()}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()